# Visualisation Results

This notebook will analyse the results coming from the GARG-AML scores, AutoAudit and Flowscope. 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# --- canonical model names (added by patch_visualisation_results.py) ---
import os, sys
_REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _REPO_ROOT not in sys.path:
    sys.path.append(_REPO_ROOT)
from src.utils.naming import (
    MODEL_DISPLAY_NAMES,
    MODEL_ORDER,
    pretty,
    pretty_many,
    gargaml_key,
)


In [ ]:
models = [
    'flowscope', 
    'autoaudit', 
    'gargaml_u',
    'gargaml_d', 
    'gargaml_tree_u',
    'gargaml_boost_u', 
    'gargaml_tree_d',
    'gargaml_boost_d'
]

patterns = [
    'laundering', 
    'separate',
    'new_mules', 
    'existing_mules'
]

In [ ]:
datasets = []

n_nodes_list = [100, 10000, 100000] # Number of nodes in the graph
m_edges_list = [1, 2, 5] # Number of edges to attach from a new node to existing nodes
p_edges_list = [0.001, 0.01] # Probability of adding an edge between two nodes
generation_method_list = [
    'Barabasi-Albert', 
    'Erdos-Renyi', 
    'Watts-Strogatz'
    ] # Generation method for the graph
n_patterns_list = [3, 5] # Number of smurfing patterns to add

for n_nodes in n_nodes_list:
        for n_patterns in n_patterns_list:
            if n_patterns <= 0.06*n_nodes:
                for generation_method in generation_method_list:
                    if generation_method == 'Barabasi-Albert':
                        p_edges = 0
                        for m_edges in m_edges_list:
                            string_name = 'synthetic_' + generation_method + '_'  + str(n_nodes) + '_' + str(m_edges) + '_' + str(p_edges) + '_' + str(n_patterns)
                            datasets.append(string_name)
                    if generation_method == 'Erdos-Renyi':
                        m_edges = 0
                        for p_edges in p_edges_list:
                            string_name = 'synthetic_' + generation_method + '_'  + str(n_nodes) + '_' + str(m_edges) + '_' + str(p_edges) + '_' + str(n_patterns)
                            datasets.append(string_name)
                    if generation_method == 'Watts-Strogatz':
                        for m_edges in m_edges_list:
                            for p_edges in p_edges_list:
                                string_name = 'synthetic_' + generation_method + '_'  + str(n_nodes) + '_' + str(m_edges) + '_' + str(p_edges) + '_' + str(n_patterns)
                                datasets.append(string_name)

In [ ]:
file = '../results-aa/synthetic_autoaudit_combined_2.csv'
df = pd.read_csv(file)
df = df.rename(columns={'Unnamed: 0': 'pattern'})
df.head()

In [ ]:
file = '../results-0/synthetic_tree_False_3.csv'
df = pd.read_csv(file)
df = df.rename(columns={'Unnamed: 0': 'pattern'})
df.head()

In [ ]:
eval(df[df['pattern']=='laundering'][datasets[0]].values[0])

In [ ]:
def gargaml_results(directed):
    if directed:
        file = '../results-0/results_performance_directed_supervised.txt'
    else:
        file = '../results-0/results_performance_undirected_supervised.txt'
    with open(file, 'r') as f:
        lines = f.readlines()
        
    results_gargaml_dict = {}
    for line in lines:
        model = line.split(':', maxsplit=1)[0].split()[0]
        results = eval(line.split(':', maxsplit=1)[1])
        results_gargaml_dict[model] = results
    return results_gargaml_dict


In [ ]:
results_flowscope = pd.read_csv('../results-0/synthetic_flowscope_combined.csv')
results_autoaudit = pd.read_csv('../results-aa/synthetic_autoaudit_combined_2.csv')
results_gargaml_undir = gargaml_results(False)
results_gargaml_dir = gargaml_results(True)
results_gargaml_tree_undir_3 = pd.read_csv('../synthetic_tree_False_3.csv')
results_gargaml_tree_undir_5 = pd.read_csv('../synthetic_tree_False_5.csv')
results_gargaml_tree_undir = results_gargaml_tree_undir_3.merge(
    results_gargaml_tree_undir_5, 
    on='Unnamed: 0', 
)
results_gargaml_tree_dir_3 = pd.read_csv('../synthetic_tree_True_3.csv')
results_gargaml_tree_dir_5 = pd.read_csv('../synthetic_tree_True_5.csv')
results_gargaml_tree_dir = results_gargaml_tree_dir_3.merge(
    results_gargaml_tree_dir_5, 
    on='Unnamed: 0', 
)

In [ ]:
results_gargaml_undir[datasets[0]]['laundering']

In [ ]:
results_flowscope.head()

In [ ]:
results_dict = {
    dataset:{
        model:{
            pattern:{
                'Precision': 0,
                'F1-score': 0,
                'AUC-ROC': 0,
                'AUC-PR': 0,
            }
            for pattern in patterns
        }
        for model in models
    }
    for dataset in datasets
}

In [ ]:
results_gargaml_tree_undir[datasets[0]]

In [ ]:
for dataset in datasets:
    for model in models:
        if model =='flowscope':
            for pattern in patterns:
                precision, f1_score, ROC, PR = tuple(eval(results_flowscope[results_flowscope['Unnamed: 0']==pattern][dataset].values[0]))
                results_dict[dataset][model][pattern]['Precision'] = precision
                results_dict[dataset][model][pattern]['F1-score'] = f1_score
                results_dict[dataset][model][pattern]['AUC-ROC'] = ROC
                results_dict[dataset][model][pattern]['AUC-PR'] = PR
        if model == 'autoaudit':
            for pattern in patterns:
                precision, f1_score, ROC, PR = tuple(eval(results_autoaudit[results_autoaudit['Unnamed: 0']==pattern][dataset].values[0]))
                results_dict[dataset][model][pattern]['Precision'] = precision
                results_dict[dataset][model][pattern]['F1-score'] = f1_score
                results_dict[dataset][model][pattern]['AUC-ROC'] = ROC
                results_dict[dataset][model][pattern]['AUC-PR'] = PR
        if model == 'gargaml_u':
            for pattern in patterns:
                try:
                    precision, f1_score, ROC, PR = tuple(results_gargaml_undir[dataset][pattern])
                except:
                    precision = f1_score = ROC = PR = 0
                results_dict[dataset][model][pattern]['Precision'] = precision
                results_dict[dataset][model][pattern]['F1-score'] = f1_score
                results_dict[dataset][model][pattern]['AUC-ROC'] = ROC
                results_dict[dataset][model][pattern]['AUC-PR'] = PR
        if model == 'gargaml_d':
            for pattern in patterns:
                try:
                    precision, f1_score, ROC, PR = tuple(results_gargaml_dir[dataset][pattern])
                except:
                    precision = f1_score = ROC = PR = 0
                results_dict[dataset][model][pattern]['Precision'] = precision
                results_dict[dataset][model][pattern]['F1-score'] = f1_score
                results_dict[dataset][model][pattern]['AUC-ROC'] = ROC
                results_dict[dataset][model][pattern]['AUC-PR'] = PR
            
        if model == 'gargaml_tree_u':
            for pattern in patterns:
                try:
                    dict_values = eval(results_gargaml_tree_undir[results_gargaml_tree_undir['Unnamed: 0']==pattern][dataset].values[0])['tree']
                    precision = dict_values['Precision']
                    f1_score = dict_values['F1']
                    ROC = dict_values['AUC_ROC']
                    PR = dict_values['AUC_PR']
                except:
                    precision = f1_score = ROC = PR = 0
                results_dict[dataset][model][pattern]['Precision'] = precision
                results_dict[dataset][model][pattern]['F1-score'] = f1_score
                results_dict[dataset][model][pattern]['AUC-ROC'] = ROC
                results_dict[dataset][model][pattern]['AUC-PR'] = PR

        if model == 'gargaml_tree_d':
            for pattern in patterns:
                try:
                    dict_values = eval(results_gargaml_tree_dir[results_gargaml_tree_dir['Unnamed: 0']==pattern][dataset].values[0])['tree']
                    precision = dict_values['Precision']
                    f1_score = dict_values['F1']
                    ROC = dict_values['AUC_ROC']
                    PR = dict_values['AUC_PR']
                except:
                    precision = f1_score = ROC = PR = 0
                results_dict[dataset][model][pattern]['Precision'] = precision
                results_dict[dataset][model][pattern]['F1-score'] = f1_score
                results_dict[dataset][model][pattern]['AUC-ROC'] = ROC
                results_dict[dataset][model][pattern]['AUC-PR'] = PR

        if model == 'gargaml_boost_u':
            for pattern in patterns:
                try:
                    dict_values = eval(results_gargaml_tree_undir[results_gargaml_tree_undir['Unnamed: 0']==pattern][dataset].values[0])['boosting']
                    precision = dict_values['Precision']
                    f1_score = dict_values['F1']
                    ROC = dict_values['AUC_ROC']
                    PR = dict_values['AUC_PR']
                except:
                    precision = f1_score = ROC = PR = 0
                results_dict[dataset][model][pattern]['Precision'] = precision
                results_dict[dataset][model][pattern]['F1-score'] = f1_score
                results_dict[dataset][model][pattern]['AUC-ROC'] = ROC
                results_dict[dataset][model][pattern]['AUC-PR'] = PR
        if model == 'gargaml_boost_d':
            for pattern in patterns:
                try:
                    dict_values = eval(results_gargaml_tree_dir[results_gargaml_tree_dir['Unnamed: 0']==pattern][dataset].values[0])['boosting']
                    precision = dict_values['Precision']
                    f1_score = dict_values['F1']
                    ROC = dict_values['AUC_ROC']
                    PR = dict_values['AUC_PR']
                except:
                    precision = f1_score = ROC = PR = 0
                results_dict[dataset][model][pattern]['Precision'] = precision
                results_dict[dataset][model][pattern]['F1-score'] = f1_score
                results_dict[dataset][model][pattern]['AUC-ROC'] = ROC
                results_dict[dataset][model][pattern]['AUC-PR'] = PR


In [ ]:
dict_values = eval(results_gargaml_tree_undir[results_gargaml_tree_undir['Unnamed: 0']==pattern][dataset].values[0])['boosting']

In [ ]:
data = []

for dataset, method_data in results_dict.items():
    for method, pattern_data in method_data.items():
        for pattern, metrics in pattern_data.items():
            data.append({
                'dataset': dataset,
                'method': method,
                'pattern': pattern,
                'performance_metric': 'Precision', 
                'performance': metrics['Precision']
            })
            
            data.append({
                'dataset': dataset,
                'method': method,
                'pattern': pattern,
                'performance_metric': 'F1-score', 
                'performance': metrics['F1-score']
            })
            
            data.append({
                'dataset': dataset,
                'method': method,
                'pattern': pattern,
                'performance_metric': 'AUC-ROC', 
                'performance': metrics['AUC-ROC']
            })

            data.append({
                'dataset': dataset,
                'method': method,
                'pattern': pattern,
                'performance_metric': 'AUC-PR', 
                'performance': metrics['AUC-PR']
            })
df = pd.DataFrame(data)
df.head(10)

In [ ]:
import seaborn as sns

In [ ]:
# Restrict to threshold-free metrics and apply canonical model display names.
df_plot = df[df['performance_metric'].isin(['AUC-ROC', 'AUC-PR'])].copy()
df_plot['method'] = df_plot['method'].map(pretty)

fig, axes = plt.subplots(2, 2, figsize=(12, 5.5), sharey=True)

for i in range(2):
    for j in range(2):
        ax = axes[i, j]
        pattern = patterns[i + 2 * j]

        sns.boxplot(
            ax=ax,
            x='performance_metric', y='performance', hue='method',
            data=df_plot[df_plot['pattern'] == pattern],
            palette='tab10', fliersize=3, showmeans=True,
            meanprops={'marker': '*', 'markerfacecolor': 'xkcd:steel',
                       'markeredgecolor': '.3', 'markersize': 10},
            medianprops={'color': 'black', 'linewidth': 3,
                         'label': '_median_'},
        )
        ax.set_title('Pattern: ' + pattern)
        ax.grid(True, which='both', ls='--', c='gray', alpha=0.3)
        # Y-label only on the left column, x-label only on the bottom row.
        ax.set_ylabel('Performance' if j == 0 else '')
        ax.set_xlabel('Performance Metric')

# Capture handles+labels once, then remove the per-axes legends.
handles, labels = axes[0, 0].get_legend_handles_labels()
for ax in axes.flat:
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()

# Single shared legend at the bottom, 4 columns x 2 rows.
fig.legend(
    handles, labels,
    loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.05),
    frameon=True, title='Method',
)
plt.tight_layout()
fig.subplots_adjust(bottom=0.18)
plt.savefig('../results/boxplot_performance_full.pdf', bbox_inches='tight')


In [ ]:
# --- Precision@K (ranking metrics, src/utils/evaluation.py) ---------------
# The ranking metrics are written into the same result cells as the legacy
# four, so they are read back from the frames already loaded above. Cells
# produced before that module landed carry only the legacy four; those come
# back as NaN and are printed as "--" in the table below.
#
# K = 10, not the module default: the 100-node datasets leave a 30-row test
# split, so k_eff = min(K, n) makes P@50 and P@100 identical there and equal to
# plain precision. K=10 is the only alert size that ranks on every tier.
K_LIST = [10]

ranking_sources = {
    'gargaml_tree_u':  (results_gargaml_tree_undir, 'tree'),
    'gargaml_boost_u': (results_gargaml_tree_undir, 'boosting'),
    'gargaml_tree_d':  (results_gargaml_tree_dir, 'tree'),
    'gargaml_boost_d': (results_gargaml_tree_dir, 'boosting'),
}

rank_rows = []
for dataset in datasets:
    for model, (frame, model_key) in ranking_sources.items():
        for pattern in patterns:
            try:
                cell = eval(frame[frame['Unnamed: 0'] == pattern][dataset].values[0])[model_key]
            except Exception:
                cell = {}
            for K in K_LIST:
                rank_rows.append({
                    'dataset': dataset,
                    'method': model,
                    'pattern': pattern,
                    'performance_metric': 'P@' + str(K),
                    'performance': cell.get('P@' + str(K), np.nan),
                })

# Idempotent: drop any P@K rows from a previous run of this cell first.
df = df[~df['performance_metric'].isin(['P@' + str(K) for K in K_LIST])]
df = pd.concat([df, pd.DataFrame(rank_rows)], ignore_index=True)
df.groupby('performance_metric')['performance'].agg(['count', 'mean'])

In [ ]:
import pandas as pd

# 1. Define your desired order exactly as you want them to appear
pattern_order = ['laundering', 'separate', 'existing_mules', 'new_mules']
pattern_titles = {                       # sub-title text (no underscores: LaTeX)
    'laundering': 'All laundering',
    'separate': 'Separate',
    'existing_mules': 'Existing mules',
    'new_mules': 'New mules',
}
method_order = ['flowscope', 'autoaudit', 'gargaml_u', 'gargaml_d', 
                'gargaml_tree_u', 'gargaml_boost_u', 'gargaml_tree_d', 'gargaml_boost_d']
metric_order = [
    'Precision',
    'F1-score',
    'AUC-ROC', 
    'AUC-PR'
    ] + ['P@' + str(K) for K in K_LIST]

# 2. Aggregation (as before)
stats = df.groupby(['pattern', 'method', 'performance_metric'])['performance'].agg(['mean', 'std']).reset_index()
stats['display'] = stats.apply(
    lambda x: '--' if pd.isna(x['mean']) else f"${x['mean']:.3f} \pm {x['std']:.3f}$", axis=1)

# 3. Pivot the table
latex_table = stats.pivot(index=['pattern', 'method'], columns='performance_metric', values='display')

# 4. CRITICAL STEP: Reorder the index and columns
# One reindex over the full product, so a (pattern, method) with no rows at all
# still yields a line in the table rather than a KeyError.
latex_table = latex_table.reindex(
    index=pd.MultiIndex.from_product([pattern_order, method_order], names=['pattern', 'method']),
    columns=metric_order,
)

# 5. Export with professional formatting
# The pattern is a full-width \multicolumn sub-title instead of its own column,
# which drops one column (and the \multirow braces) from the width.
n_cols = 1 + len(metric_order)           # method column + one column per metric
lines = [
    r'\begin{table}[t]',
    r'\centering',
    r'\caption{Performance Comparison Across Patterns}',
    r'\label{tab:results}',
    r'\begin{tabular}{l' + 'c' * len(metric_order) + '}',
    r'\toprule',
    ' & '.join(['Method'] + metric_order) + r' \\',
]
for pattern in pattern_order:
    lines += [
        r'\midrule',
        r'\multicolumn{%d}{l}{\textit{%s}} \\' % (n_cols, pattern_titles[pattern]),
        r'\midrule',
    ]
    for method in method_order:
        cells = ['--' if pd.isna(v) else v for v in latex_table.loc[(pattern, method), metric_order]]
        lines.append(' & '.join([pretty(method)] + cells) + r' \\')
lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']

latex_code = '\n'.join(lines)
print(latex_code)

In [ ]:
for n_nodes in n_nodes_list:
    fig, axes = plt.subplots(2, 2, figsize=(15, 6))
    for i in range(2):
        for j in range(2): 
            pattern = patterns[i+2*j]

            sns.boxplot(ax=axes[i,j], x='performance_metric', y='performance', hue='method', data=df[(df['pattern']==pattern)&(df['dataset'].str.contains('_'+str(n_nodes)+'_'))], palette='tab10',showmeans=True, meanprops={'marker':'*', 'markerfacecolor':'xkcd:steel', 'markeredgecolor':'.3', 'markersize': 10}, medianprops={'color': 'black', 'linewidth':2,'label': '_median_', 'linewidth':3})
            axes[i,j].set_title('Performance for the different methods - Pattern: '+ pattern)
            axes[i,j].set_xlabel('Performance Metric')
            axes[i,j].set_ylabel('Value')
            axes[i,j].legend(title='method', bbox_to_anchor=(1.0, 1), loc='upper left')
            axes[i,j].grid(True,which="both",ls="--",c='gray', alpha=0.3)

    fig.suptitle(f'Performance Analysis for #Nodes: {n_nodes}', fontsize=16)
    plt.tight_layout()  # Adjust layout to fit the global title

    plt.savefig('../results/boxplot_performance_'+str(n_nodes)+'_full.pdf')

In [ ]:
def give_order(models, pattern, dataset):
    # Initialize a dictionary to store the results
    results = {model: {'AUC-ROC': 0, 'AUC-PR': 0} for model in models}

    for model in models:
        results[model]['AUC-ROC'] = results_dict[dataset][model][pattern]['AUC-ROC']
        results[model]['AUC-PR'] = results_dict[dataset][model][pattern]['AUC-PR']

    sorted_models_ROC = []
    sorted_models_PR = []

    sorted_models_ROC_ = sorted(models, key=lambda model: results[model]['AUC-ROC'], reverse=True)
    sorted_models_PR_ = sorted(models, key=lambda model: results[model]['AUC-PR'], reverse=True)
    sorted_models_ROC.append(sorted_models_ROC_)
    sorted_models_PR.append(sorted_models_PR_)

    return sorted_models_ROC, sorted_models_PR

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True, sharey=True)

for idx, pattern in enumerate(patterns):
    ax = axes[idx // 2, idx % 2]

    sorted_models_ROC_list = []
    sorted_models_PR_list = []

    for dataset in datasets:
        order_ROC, order_PR = give_order(models, pattern, dataset)
        for i in order_ROC:
            sorted_models_ROC_list.append(i)
        for i in order_PR:
            sorted_models_PR_list.append(i)

    dict_order_ROC = {}
    dict_order_PR = {}

    for model in models:
        dict_order_ROC[model] = []
        dict_order_PR[model] = []
        for i in sorted_models_ROC_list:
            dict_order_ROC[model].append(i.index(model) + 1)
        for i in sorted_models_PR_list:
            dict_order_PR[model].append(i.index(model) + 1)

    combined_data = []
    for model in models:
        method_label = pretty(model)
        for value in dict_order_ROC[model]:
            combined_data.append({'Method': method_label, 'Rank': value, 'Metric': 'AUC-ROC'})
        for value in dict_order_PR[model]:
            combined_data.append({'Method': method_label, 'Rank': value, 'Metric': 'AUC-PR'})

    combined_df = pd.DataFrame(combined_data)

    sns.boxplot(
        ax=ax, x='Method', y='Rank', hue='Metric', data=combined_df,
        palette='tab10', fliersize=3, showmeans=True,
        medianprops={'color': 'black', 'linewidth': 3, 'label': '_median_'},
        meanprops={'marker': '*', 'markerfacecolor': 'xkcd:steel',
                   'markeredgecolor': '.3', 'markersize': 10},
    )
    ax.set_title('Pattern: ' + pattern)
    ax.grid(True, which='both', ls='--', c='gray', alpha=0.3)
    # Y-label only on left column, x-label only on bottom row.
    ax.set_ylabel('Rank' if idx % 2 == 0 else '')
    ax.set_xlabel('')

# Rotate x-tick labels on the bottom row (sharex hides them on the top row).
for ax in axes[1, :]:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# Single shared legend at the top (only 2 entries; keeps it out of the
# way of the rotated x-tick labels on the bottom row).
handles, labels = axes[0, 0].get_legend_handles_labels()
for ax in axes.flat:
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()
fig.legend(
    handles, labels,
    loc='upper center', ncol=2, bbox_to_anchor=(0.5, 0.99),
    frameon=True, title='Metric',
)
# Reserve ~8% at the top for the legend and let tight_layout handle the
# rotated x-tick labels at the bottom.
fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.savefig('../results/boxplot_rank_full.pdf', bbox_inches='tight')


In [ ]:
for n_nodes in n_nodes_list:
    datasets_nodes = [dataset for dataset in datasets if dataset.split('_')[2] == str(n_nodes)]    
    fig, axes = plt.subplots(2, 2, figsize=(15, 6))

    for idx, pattern in enumerate(patterns):
        sorted_models_ROC_list = []
        sorted_models_PR_list = []

        for dataset in datasets_nodes:
            order_ROC, order_PR = give_order(models, pattern, dataset)
            for i in order_ROC:
                sorted_models_ROC_list.append(i)
            for i in order_PR:
                sorted_models_PR_list.append(i)
        
        dict_order_ROC = {}
        dict_order_PR = {}

        for model in models:
            dict_order_ROC[model] = []
            dict_order_PR[model] = []
            for i in sorted_models_ROC_list:
                dict_order_ROC[model].append(i.index(model) + 1)
            for i in sorted_models_PR_list:
                dict_order_PR[model].append(i.index(model) + 1)

        combined_data = []
        for model in models:
            for value in dict_order_ROC[model]:
                combined_data.append({'Method': model, 'Rank': value, 'Metric': 'AUC-ROC'})
            for value in dict_order_PR[model]:
                combined_data.append({'Method': model, 'Rank': value, 'Metric': 'AUC-PR'})
        
        combined_df = pd.DataFrame(combined_data)

        # Plot the boxplot
        sns.boxplot(ax=axes[idx//2, idx%2], x='Method', y='Rank', hue='Metric', data=combined_df, palette='tab10', showmeans=True, medianprops={'color': 'black', 'linewidth':2,'label': '_median_', 'linewidth':3}, meanprops={'marker':'*', 'markerfacecolor':'xkcd:steel', 'markeredgecolor':'.3', 'markersize': 10})
        axes[idx//2, idx%2].set_title('Rank of the methods - Pattern: '+ pattern)
        axes[idx//2, idx%2].set_xlabel('Method')
        axes[idx//2, idx%2].set_ylabel('Rank')
        axes[idx//2, idx%2].legend(title='Metric', bbox_to_anchor=(1.0, 1), loc='upper left')
        axes[idx//2, idx%2].grid(True,which="both",ls="--",c='gray', alpha=0.3)  
    fig.suptitle(f'Rank Analysis for #Nodes: {n_nodes}', fontsize=16)
    plt.tight_layout()
    plt.savefig('boxplot_rank_'+str(n_nodes)+'_full.pdf')


## Statistical test of the ranks

Two statistical tests are applied. First, the Friedman test is used to test if there are statistically significant differences in the mean ranks of the methods. If there is, we apply the post-hoc Nemenyi test to compare the methods two-by-two and to construct the critical distance diagrams. 

In [ ]:
import numpy as np
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp
import matplotlib.pyplot as plt

### Friedman Test

In [ ]:
# Example scores of 6 classifiers over 5 datasets
# Rows = datasets, Columns = models
scores = np.array([
    [3.00, 2.00, 4.00, 5.00, 1.00],
    [2.00, 2.00, 2.00, 2.00, 2.00],
    [5.00, 3.00, 4.00, 2.00, 1.00],
    [3.00, 1.00, 5.00, 4.00, 2.00],
    [1.00, 2.00, 4.00, 3.00, 5.00],
    [2.00, 2.00, 2.00, 3.00, 1.00],
])

# Run the Friedman test
stat, p = friedmanchisquare(*scores.T)

print(f"Friedman test statistic: {stat:.4f}, p-value: {p:.4f}")

if p < 0.2:
    print("Significant differences found — proceeding with Critical Distance Diagram")

    # Compute ranks (1 = best, higher = worse)
    ranks = np.argsort(np.argsort(scores, axis=1), axis=1) + 1  # ascending order

    # Average ranks per model
    avg_ranks = np.mean(ranks, axis=0)
    print("Average ranks:", avg_ranks)

    nf = sp.posthoc_nemenyi_friedman(scores)
    nf.index=['Evolutions in AI','Explainable AI','LLMs & GenAI', 'AI Act', 'Panel']
    nf.columns=['Evolutions in AI','Explainable AI','LLMs & GenAI', 'AI Act', 'Panel']
    # Format: diagonal, non-significant, p<0.001, p<0.01, p<0.05
    cmap = ['1', '#fb6a4a',  '#08306b',  '#4292c6', '#c6dbef']
    heatmap_args = {'cmap': cmap, 'linewidths': 0.25, 'linecolor': '0.5', 'square': True}
    sp.sign_plot(nf, **heatmap_args)

else:
    print("No statistically significant differences found — skipping post-hoc analysis.")


In [ ]:
avg_ranks_dict = {}
for i, model in enumerate(nf.index):
    avg_ranks_dict[model] = avg_ranks[i]
avg_ranks_dict
plt.figure(figsize=(10, 2))
plt.title('Critical difference diagram of average score ranks')
sp.critical_difference_diagram(avg_ranks_dict, nf)

In [ ]:
df_selection = df[(df['pattern']=='laundering')&(df['performance_metric']=='AUC-ROC')][['dataset', 'method', 'performance']]
avg_rank = df_selection.groupby('dataset').performance.rank(ascending=False).groupby(df_selection.method).mean()
avg_rank

In [ ]:
df_selection

In [ ]:
dict_data = {}
for method in models:
    dict_data[method] = []
    for dataset in datasets:
        dict_data[method].append(results_dict[dataset][method]['laundering']['AUC-ROC'])

In [ ]:
df_selection.groupby('dataset').performance.rank(ascending=False).groupby(df_selection.method).mean()

In [ ]:
scores = np.array(list(dict_data.values())).T

# Run the Friedman test
stat, p = friedmanchisquare(*scores.T)

print(f"Friedman test statistic: {stat:.4f}, p-value: {p:.4f}")

if p<0.05:
    print("Significant differences found — proceeding with Critical Distance Diagram")

    # Compute ranks (1 = best, higher = worse)
    ranks = np.argsort(np.argsort(-scores, axis=1), axis=1) + 1  # descending order

    # Average ranks per model
    avg_ranks = np.mean(ranks, axis=0)
    print("Average ranks:", avg_ranks)

    nf = sp.posthoc_nemenyi_friedman(scores)
    model_names = list(dict_data.keys())
    nf.index=nf.columns=model_names
    
    # Format: diagonal, non-significant, p<0.001, p<0.01, p<0.05
    cmap = ['1', '#fb6a4a',  '#08306b',  '#4292c6', '#c6dbef']
    heatmap_args = {'cmap': cmap, 'linewidths': 0.25, 'linecolor': '0.5', 'square': True}
    sp.sign_plot(nf, **heatmap_args)

else:
    print("No statistically significant differences found — skipping post-hoc analysis.")


In [ ]:
# Define the colors for the models based on their order in the boxplot
model_colors = {
    'flowscope': 'tab:blue',
    'autoaudit': 'tab:orange',
    'gargaml_u': 'tab:green',
    'gargaml_d': 'tab:red', 
    'gargaml_tree_u': 'tab:purple',
    'gargaml_boost_u': 'tab:brown', 
    'gargaml_tree_d': 'tab:pink',
    'gargaml_boost_d': 'tab:gray'
}

pretty_model_names = dict(MODEL_DISPLAY_NAMES)



avg_ranks_dict = dict(df_selection.groupby('dataset').performance.rank(ascending=False).groupby(df_selection.method).mean())

# --- IMPORTANT: Create a new dictionary with PRETTY labels ---
avg_ranks_pretty = {pretty_model_names[model]: rank for model, rank in avg_ranks_dict.items()}
nf.rename(index=pretty_model_names, columns=pretty_model_names, inplace=True)

# Also adapt the color palette using pretty labels
color_palette_pretty = {pretty_model_names[model]: model_colors[model] for model in avg_ranks_dict.keys()}

# Now plot with pretty names
plt.figure(figsize=(10, 2))
plt.title('Critical Difference Diagram of Average Score Ranks')
sp.critical_difference_diagram(
    ranks=avg_ranks_pretty,
    #ranks=avg_ranks_dict,
    sig_matrix=nf,  # Assuming nf matches the order of avg_ranks_dict
    label_fmt_left='{label} [{rank:.2f}]  ',
    label_fmt_right='  [{rank:.2f}] {label}',
    color_palette=color_palette_pretty,
    #color_palette={model: model_colors[model] for model in avg_ranks_dict.keys()}
)

In [ ]:
friedman_stats_roc = []
nemenyi_matrices_roc = {}

fig, axes = plt.subplots(4, 1, figsize=(8, 6.5))

for i_ax in range(4):
    pattern = patterns[i_ax]

    df_selection = df[(df['pattern']==pattern)&(df['performance_metric']=='AUC-ROC')][['dataset', 'method', 'performance']]
    avg_rank = df_selection.groupby('dataset').performance.rank(ascending=False).groupby(df_selection.method).mean()

    dict_data = {}
    for method in models:
        dict_data[method] = []
        for dataset in datasets:
            dict_data[method].append(results_dict[dataset][method][pattern]['AUC-ROC'])        

    scores = np.array(list(dict_data.values())).T

    # Run the Friedman test
    stat, p = friedmanchisquare(*scores.T)

    print(f"Friedman test statistic: {stat:.4f}, p-value: {p:.4f}")
    friedman_stats_roc.append({
        'pattern': pattern,
        'chi2': float(stat),
        'p_value': float(p),
        'df': scores.shape[1] - 1,   # k - 1
        'N_datasets': scores.shape[0],
        'k_methods': scores.shape[1],
    })

    if p<0.05:
        print("Significant differences found — proceeding with Critical Distance Diagram")

        # Compute ranks (1 = best, higher = worse)
        ranks = np.argsort(np.argsort(-scores, axis=1), axis=1) + 1  # descending order

        # Average ranks per model
        avg_ranks = np.mean(ranks, axis=0)
        print("Average ranks:", avg_ranks)

        nf = sp.posthoc_nemenyi_friedman(scores)
        model_names = list(dict_data.keys())
        nf.index=nf.columns=model_names
        nemenyi_matrices_roc[pattern] = nf.copy()

        avg_ranks_dict = dict(df_selection.groupby('dataset').performance.rank(ascending=False).groupby(df_selection.method).mean())

        # --- IMPORTANT: Create a new dictionary with PRETTY labels ---
        avg_ranks_pretty = {pretty_model_names[model]: rank for model, rank in avg_ranks_dict.items()}
        nf.rename(index=pretty_model_names, columns=pretty_model_names, inplace=True)

        # Also adapt the color palette using pretty labels
        color_palette_pretty = {pretty_model_names[model]: model_colors[model] for model in avg_ranks_dict.keys()}
        
        axes[i_ax].set_title('Pattern: '+pattern)
        sp.critical_difference_diagram(
            ranks=avg_ranks_pretty,
            # ranks=avg_ranks_dict,
            sig_matrix=nf,
            label_fmt_left='{label} ({rank:.2f})  ',
            label_fmt_right='  ({rank:.2f}) {label}',
            label_props={'fontweight': 'bold'},
            color_palette= color_palette_pretty, 
            # color_palette={model: model_colors[model] for model in avg_ranks_dict.keys()},
            ax=axes[i_ax]
            )
        
        # Get correct handles and labels
        handles, labels = axes[i_ax].get_legend_handles_labels()

    else:
        print("No statistically significant differences found — skipping post-hoc analysis.")

plt.suptitle('Critical difference diagram for average rank according to the AUC-ROC')
plt.tight_layout()
plt.savefig('../results/CD_ROC_full.pdf')

In [ ]:
friedman_stats_pr = []
nemenyi_matrices_pr = {}

fig, axes = plt.subplots(4, 1, figsize=(8, 6.5))

for i_ax in range(4):
    pattern = patterns[i_ax]

    df_selection = df[(df['pattern']==pattern)&(df['performance_metric']=='AUC-PR')][['dataset', 'method', 'performance']]
    avg_rank = df_selection.groupby('dataset').performance.rank(ascending=False).groupby(df_selection.method).mean()

    dict_data = {}
    for method in models:
        dict_data[method] = []
        for dataset in datasets:
            dict_data[method].append(results_dict[dataset][method][pattern]['AUC-PR'])        

    scores = np.array(list(dict_data.values())).T

    # Run the Friedman test
    stat, p = friedmanchisquare(*scores.T)

    print(f"Friedman test statistic: {stat:.4f}, p-value: {p:.4f}")
    friedman_stats_pr.append({
        'pattern': pattern,
        'chi2': float(stat),
        'p_value': float(p),
        'df': scores.shape[1] - 1,   # k - 1
        'N_datasets': scores.shape[0],
        'k_methods': scores.shape[1],
    })

    if p<0.05:
        print("Significant differences found — proceeding with Critical Distance Diagram")

        # Compute ranks (1 = best, higher = worse)
        ranks = np.argsort(np.argsort(-scores, axis=1), axis=1) + 1  # descending order

        # Average ranks per model
        avg_ranks = np.mean(ranks, axis=0)
        print("Average ranks:", avg_ranks)

        nf = sp.posthoc_nemenyi_friedman(scores)
        model_names = list(dict_data.keys())
        nf.index=nf.columns=model_names
        nemenyi_matrices_pr[pattern] = nf.copy()

        avg_ranks_dict = dict(df_selection.groupby('dataset').performance.rank(ascending=False).groupby(df_selection.method).mean())

        # --- IMPORTANT: Create a new dictionary with PRETTY labels ---
        avg_ranks_pretty = {pretty_model_names[model]: rank for model, rank in avg_ranks_dict.items()}
        nf.rename(index=pretty_model_names, columns=pretty_model_names, inplace=True)

        # Also adapt the color palette using pretty labels
        color_palette_pretty = {pretty_model_names[model]: model_colors[model] for model in avg_ranks_dict.keys()}

        axes[i_ax].set_title('Pattern: '+pattern)
        sp.critical_difference_diagram(
            ranks=avg_ranks_pretty,
            # ranks=avg_ranks_dict,
            sig_matrix=nf,
            label_fmt_left='{label} ({rank:.2f})  ',
            label_fmt_right='  ({rank:.2f}) {label}',
            label_props={'fontweight': 'bold'},
            color_palette= color_palette_pretty, 
            # color_palette={model: model_colors[model] for model in avg_ranks_dict.keys()},
            ax=axes[i_ax]
            )
    else:
        print("No statistically significant differences found — skipping post-hoc analysis.")

plt.suptitle('Critical difference diagram for average rank according to the AUC-PR')
plt.tight_layout()
plt.savefig('../results/CD_PR_full.pdf')

## Friedman + Nemenyi: statistics for the appendix

The cells above run the Friedman test (Friedman 1937, 1940) once per pattern
&times; metric and plot the post-hoc Nemenyi (Nemenyi 1963) Critical Difference
diagram. This section consolidates the test statistics so the values can be
quoted directly in the paper.

**Method (matches the implementation):**

1. For each of the 4 injection patterns &times; 2 threshold-free metrics
   (AUC-ROC, AUC-PR), we collect the performance of the $k = 8$ methods on the
   $N = 66$ synthetic datasets, giving an $N \times k$ score matrix.
2. We run `scipy.stats.friedmanchisquare` on the matrix. Under $H_0$ (all
   methods are equivalent), the statistic is approximately $\chi^2$ with
   $k - 1 = 7$ degrees of freedom.
3. We control family-wise Type I error across the 8 Friedman tests with
   **Bonferroni** (multiply each $p$ by 8) and **Holm**'s sequentially-rejective
   procedure. The Holm-adjusted $p$ is the more powerful of the two; reject
   $H_0$ when it is below $\alpha = 0.05$.
4. When $H_0$ is rejected, we run the post-hoc **Nemenyi test** via
   `scikit_posthocs.posthoc_nemenyi_friedman`, which returns a $k \times k$
   matrix of pairwise $p$-values.
5. The **Critical Distance** for the Nemenyi CD diagram is
   $\mathrm{CD} = q_\alpha \sqrt{\frac{k(k+1)}{6N}}$
   where $q_\alpha$ is the studentized range statistic divided by
   $\sqrt{2}$ (Demšar 2006, Table 5). For $k=8$ and $\alpha=0.05$,
   $q_{0.05} = 3.031$, giving $\mathrm{CD} \approx 1.29$ for $N=66$.

**How the CD-diagram cliques are derived.**
Two methods are connected by a horizontal bar (a *clique*) when the absolute
difference in their average ranks is below the Critical Distance, equivalently
when their Nemenyi pairwise $p$-value exceeds $\alpha$. The cliques are the
connected components of the graph $(V, E)$ with $V$ = methods and
$E = \{(i, j) : p_{ij} > \alpha\}$; `scikit_posthocs.critical_difference_diagram`
draws one horizontal segment per connected component, ordered from lowest
(best) rank on the left to highest on the right.

Note that the Nemenyi test inherently controls Type I error across all
$\binom{k}{2} = 28$ pairwise comparisons by using the studentized range
distribution, so no additional adjustment is needed beyond the CD threshold.


In [ ]:
# --- Friedman/Nemenyi appendix (added by patch_friedman_reporting.py) ---
import numpy as np
import pandas as pd

# -- 1. Aggregate Friedman stats from cells 36 (AUC-ROC) and 37 (AUC-PR) --
rows = []
for stats_list, metric in [(friedman_stats_roc, "AUC-ROC"),
                           (friedman_stats_pr,  "AUC-PR")]:
    for s in stats_list:
        rows.append({**s, "metric": metric})
friedman_df = pd.DataFrame(rows)[
    ["metric", "pattern", "chi2", "df", "N_datasets", "k_methods", "p_value"]
]

# -- 2. Multiple-testing correction across the 8 (pattern x metric) tests --
m = len(friedman_df)
p_raw = friedman_df["p_value"].to_numpy()

# Bonferroni: simple multiplication, capped at 1.0
friedman_df["p_bonferroni"] = np.minimum(p_raw * m, 1.0)

# Holm: order p ascending, multiply by (m - rank), enforce monotonicity
order = np.argsort(p_raw)
p_holm = np.empty_like(p_raw)
running_max = 0.0
for rank, idx in enumerate(order):
    p_adj = (m - rank) * p_raw[idx]
    running_max = max(running_max, min(p_adj, 1.0))
    p_holm[idx] = running_max
friedman_df["p_holm"] = p_holm

friedman_df["reject_H0_holm_005"] = friedman_df["p_holm"] < 0.05

# -- 3. Nemenyi Critical Distance --
# Try scipy's studentized_range (scipy >= 1.7); fall back to Demsar 2006 Table 5
# for alpha = 0.05.
DEMSAR_Q_ALPHA_005 = {
    2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850,
    7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164,
}
k = int(friedman_df["k_methods"].iloc[0])
N = int(friedman_df["N_datasets"].iloc[0])
alpha = 0.05
try:
    from scipy.stats import studentized_range
    q_alpha = studentized_range.ppf(1 - alpha, k, np.inf) / np.sqrt(2)
    q_source = "scipy.stats.studentized_range"
except (ImportError, AttributeError):
    q_alpha = DEMSAR_Q_ALPHA_005[k]
    q_source = "Demsar 2006 Table 5 (hardcoded fallback)"
CD = q_alpha * np.sqrt(k * (k + 1) / (6.0 * N))

print(f"Nemenyi CD parameters: k={k}, N={N}, alpha={alpha}")
print(f"  q_alpha = {q_alpha:.4f}  (source: {q_source})")
print(f"  CD      = {CD:.4f}\n")

friedman_df["CD_nemenyi"] = CD

# -- 4. Save: per-test CSV + per-pattern Nemenyi matrices --
import os
os.makedirs("../results", exist_ok=True)
friedman_df.to_csv("../results/friedman_results.csv", index=False)
print("wrote ../results/friedman_results.csv")

for metric, nf_dict in [("AUC-ROC", nemenyi_matrices_roc),
                        ("AUC-PR",  nemenyi_matrices_pr)]:
    for pattern, nf_mat in nf_dict.items():
        path = f"../results/nemenyi_pvalues_{metric}_{pattern}.csv"
        # Rename via canonical display names so the appendix tables are
        # consistent with Tables 9-11 and Figures 8-9.
        nf_disp = nf_mat.rename(index=pretty, columns=pretty)
        nf_disp.to_csv(path)
print(f"wrote {len(nemenyi_matrices_roc) + len(nemenyi_matrices_pr)} Nemenyi matrices")

# -- 5. Paper-ready table (printed; copy-paste into the appendix) --
display_cols = ["metric", "pattern", "chi2", "df", "p_value",
                "p_bonferroni", "p_holm", "reject_H0_holm_005"]
print("\n=== Friedman test summary (k={}, N={}) ===".format(k, N))
print(friedman_df[display_cols].to_string(index=False,
                                          formatters={
                                              "chi2": "{:.3f}".format,
                                              "p_value": "{:.3e}".format,
                                              "p_bonferroni": "{:.3e}".format,
                                              "p_holm": "{:.3e}".format,
                                          }))

# LaTeX version
print("\n=== LaTeX table ===")
latex_df = friedman_df[display_cols].copy()
latex_df.columns = ["Metric", "Pattern", r"$\chi^2$", r"df", r"$p$",
                    r"$p_{\text{Bonf}}$", r"$p_{\text{Holm}}$",
                    r"Reject $H_0$ (Holm, $\alpha=0.05$)"]
print(latex_df.to_latex(index=False, escape=False,
                        float_format=lambda x: f"{x:.3e}" if x < 0.01 else f"{x:.3f}"))


# IBM Dataset

In this part of the notebook, we analyse the results for the IBM data set. 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# GARG-AML Undirected
file = '../results-0/results_performance_IBM_undirected.txt'
with open(file, 'r') as f:
    lines = f.readlines()

model_list = []
dataset_list = []
pattern_list = []
cutoff_list = []
precision_list = []
f1_score_list = []
AUCROC_list = []
AUCPR_list = []

for line in lines:
    long_split = line.strip().split('_')
    model = pretty(gargaml_key('base', directed=False))
    dataset = long_split[0]
    pattern = long_split[1]
    cutoff_results = long_split[2].split(': ', maxsplit=3)
    cutoff = cutoff_results[0].split(' ')[0]
    try:
        results = eval(cutoff_results[-1])
    except:
        results = [0, 0, 0, 0]
    precision = results[0]
    f1_score = results[1]
    AUCROC = results[2]
    AUCPR = results[3]
    model_list.append(model)
    dataset_list.append(dataset)
    pattern_list.append(pattern)
    cutoff_list.append(cutoff)
    precision_list.append(precision)
    f1_score_list.append(f1_score)
    AUCROC_list.append(AUCROC)
    AUCPR_list.append(AUCPR)


In [ ]:
# GARG-AML Directed
file = '../results-0/results_performance_IBM_directed.txt'
with open(file, 'r') as f:
    lines = f.readlines()

for line in lines:
    long_split = line.strip().split('_')
    model = pretty(gargaml_key('base', directed=True))
    dataset = long_split[0]
    pattern = long_split[1]
    cutoff_results = long_split[2].split(': ', maxsplit=3)
    cutoff = cutoff_results[0].split(' ')[0]
    try:
        results  = eval(cutoff_results[-1])
    except:
        results = [0, 0, 0, 0]
    precision = results[0]
    f1_score = results[1]
    AUCROC = results[2]
    AUCPR = results[3]
    model_list.append(model)
    dataset_list.append(dataset)
    pattern_list.append(pattern)
    cutoff_list.append(cutoff)
    precision_list.append(precision)
    f1_score_list.append(f1_score)
    AUCROC_list.append(AUCROC)
    AUCPR_list.append(AUCPR)

In [ ]:
# Flowscope HI-Small
file = '../results/flowscope_performance_HI-Small.txt'
with open(file, 'r') as f:
    lines = f.readlines()

for line in lines:
    long_split = line.strip().split('_')
    model = pretty('flowscope')
    dataset = 'HI-Small'
    pattern = long_split[1]
    cutoff_results = long_split[3].split(' ', maxsplit=3)
    cutoff = cutoff_results[0]
    try:
        results  = eval(cutoff_results[-1])
    except:
        results = [0, 0]
    AUCROC = results[0]
    AUCPR = results[1]
    model_list.append(model)
    dataset_list.append(dataset)
    pattern_list.append(pattern)
    cutoff_list.append(cutoff)
    # FlowScope reports only AUC-ROC / AUC-PR; precision and F1 are not measured.
    precision_list.append(np.nan)
    f1_score_list.append(np.nan)
    AUCROC_list.append(AUCROC)
    AUCPR_list.append(AUCPR)

In [ ]:
# Flowscope LI-Large
file = '../results/flowscope_performance_LI-Large.txt'
with open(file, 'r') as f:
    lines = f.readlines()

for line in lines:
    long_split = line.strip().split('_')
    print(long_split)
    model = pretty('flowscope')
    dataset = 'LI-Large'
    pattern = long_split[1]
    cutoff_results = long_split[3].split(' ', maxsplit=3)
    cutoff = cutoff_results[0]
    try:
        results  = eval(cutoff_results[-1])
    except:
        results = [0, 0]
    AUCROC = results[0]
    AUCPR = results[1]
    model_list.append(model)
    dataset_list.append(dataset)
    pattern_list.append(pattern)
    cutoff_list.append(cutoff)
    # FlowScope reports only AUC-ROC / AUC-PR; precision and F1 are not measured.
    precision_list.append(np.nan)
    f1_score_list.append(np.nan)
    AUCROC_list.append(AUCROC)
    AUCPR_list.append(AUCPR)

In [ ]:
models_IBM_tree = ['tree', 'boosting']
directed_list = ['undirected', 'directed']
IMB_data_list = ['HI-Small', 'LI-Large']
metrics_list = ['precision', 'f1', 'AUC_ROC', 'AUC_PR']
IBM_cutoffs = [0.0, 0.1, 0.2, 0.3, 0.5, 0.9] # matches src/utils/evaluation.py::CUT_OFFS
IBM_patterns = ['Is Laundering', 'FAN-OUT', 'FAN-IN', 'GATHER-SCATTER', 'SCATTER-GATHER', 'CYCLE', 'RANDOM', 'BIPARTITE', 'STACK']

# 0.0 ("at least one laundering transaction") joined the sweep during the
# revision, so a matrix written before that has no 0.0 row. A missing row is
# read as NaN and reported below rather than raising, so this notebook runs
# against old and new result files alike -- but a table full of NaN at 0.0
# means the model scripts still have to be re-run, not that the cell is empty.
missing = []

for ds in IMB_data_list:
    for d in directed_list:
        for m in models_IBM_tree:
            for cut_off in IBM_cutoffs:
                for pattern in IBM_patterns:
                    variant = 'boost' if m == 'boosting' else m
                    model_list.append(pretty(gargaml_key(variant, directed=(d == 'directed'))))
                    dataset_list.append(ds)
                    pattern_list.append(pattern)
                    cutoff_list.append(str(cut_off))
                    for pm in metrics_list:
                        file = '../results/'+ds+'_'+pm+'_'+m+'_'+d+'_combined.csv'
                        df = pd.read_csv(file, index_col=0)
                        try:
                            value = df.loc[cut_off][pattern]
                        except KeyError:
                            value = np.nan
                            missing.append((ds, d, m, pm, cut_off))
                        if pm == 'precision':
                            precision_list.append(value)
                        elif pm == 'f1':
                            f1_score_list.append(value)
                        elif pm == 'AUC_ROC':
                            AUCROC_list.append(value)
                        else:
                            AUCPR_list.append(value)

if missing:
    cuts = sorted({c for *_, c in missing})
    print(str(len(missing))+' cells missing from the result matrices, at cut-off(s) '
          +str(cuts)+' -- re-run scripts/gargaml_tree.py to fill them.')

In [ ]:
results_df = pd.DataFrame({
    'model': model_list,
    'dataset': dataset_list,
    'pattern': pattern_list,
    'cutoff': cutoff_list,
    'Precision': precision_list,
    'F1-score': f1_score_list,
    'AUC-ROC': AUCROC_list,
    'AUC-PR': AUCPR_list
})

In [ ]:
results_df.fillna(0, inplace=True)

In [ ]:
# --- Precision@K for the IBM data ----------------------------------------
# K = 1000 here: the IBM test splits are large (154k accounts on HI-Small), so
# a realistic alert queue is the binding constraint rather than the split size.
# The ranking metrics are not in the <metric>_<model>_<direction>_combined.csv
# matrices read above -- those carry the legacy four only. They live in the
# tidy long-form file src/utils/evaluation.py writes beside them, one row per
# (model, cutoff, target, metric, K).
K_IBM = 1000

pak_frames = []
for ds in IMB_data_list:
    for d in directed_list:
        path = '../results/' + ds + '_' + d + '_metrics.csv'
        if not os.path.exists(path):
            print('no tidy metrics for', ds, d, '-- P@%d will be "--"' % K_IBM)
            continue
        tidy = pd.read_csv(path)
        tidy = tidy[(tidy['metric'] == 'P@K') & (tidy['K'] == K_IBM)]
        pak_frames.append(pd.DataFrame({
            'model': tidy['model'].map(pretty),
            'dataset': tidy['dataset'],
            'pattern': tidy['target'],
            'cutoff': tidy['cutoff'].astype(str),
            'P@%d' % K_IBM: tidy['value'],
        }))

pak_df = pd.concat(pak_frames, ignore_index=True) if pak_frames else pd.DataFrame(
    columns=['model', 'dataset', 'pattern', 'cutoff', 'P@%d' % K_IBM])

results_df = results_df.drop(columns=['P@%d' % K_IBM], errors='ignore').merge(
    pak_df, on=['model', 'dataset', 'pattern', 'cutoff'], how='left')
results_df.groupby('dataset')['P@%d' % K_IBM].agg(['count', 'max'])

In [ ]:
results_df.head()

In [ ]:
results_plot_df = results_df[(results_df['dataset']==dataset) & (results_df['pattern'].isin(['Is Laundering', 'GATHER-SCATTER', 'SCATTER-GATHER'])) & (results_df['cutoff'].isin(['0.0', '0.1', '0.5' ,'0.9']))]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

datasets = ['HI-Small', 'LI-Large']
for dataset in datasets:
    results_plot_df = results_df[(results_df['dataset']==dataset) & (results_df['pattern'].isin(['Is Laundering', 'GATHER-SCATTER', 'SCATTER-GATHER'])) & (results_df['cutoff'].isin(['0.0', '0.1', '0.5' ,'0.9']))]

    # Ensure consistent aesthetics
    sns.set(style="whitegrid")

    # Assume results_plot_df is already defined
    # Extract unique patterns and cutoffs
    patterns = results_plot_df['pattern'].unique()
    cutoffs = results_plot_df['cutoff'].unique()
    models = [pretty(k) for k in [
        'flowscope',
        'gargaml_u', 'gargaml_d',
        'gargaml_tree_u', 'gargaml_boost_u',
        'gargaml_tree_d', 'gargaml_boost_d',
    ]]
    palette = sns.color_palette("tab10", len(models)+1)
    del (palette[1])

    # Create a mapping from model names to colors
    model_color_map = dict(zip(models, palette))

    # Create the subplot grid
    fig, axes = plt.subplots(len(patterns), len(cutoffs), figsize=(6.5 * len(cutoffs), 3 * len(patterns)), squeeze=False)

    for i, pattern in enumerate(patterns):
        for j, cutoff in enumerate(cutoffs):
            ax = axes[i, j]
            subset = results_plot_df[(results_plot_df['pattern'] == pattern) & (results_plot_df['cutoff'] == cutoff)]
            ylim_max = subset['AUC-ROC'].max()*1.1
            ax.set_ylim(0, ylim_max)
            # Primary bar plot for AUC-ROC
            for idx, model in enumerate(models):
                model_data = subset[subset['model'] == model]
                roc = model_data['AUC-ROC'].values[0]
                ax.bar(idx - 0.2, model_data['AUC-ROC'], width=0.4, label=model if i == 0 and j == 0 else "", 
                    color=model_color_map[model], edgecolor='black')
                ax.text(idx - 0.23, roc-0.001, f"{roc*100:.1f}%", ha='center', va='bottom', fontsize=8)
            
            # Twin axis for AUC-PR
            ylim_max_2 = subset['AUC-PR'].max()*1.1
            ax2 = ax.twinx()
            ax2.set_ylim(0, ylim_max_2)
            for idx, model in enumerate(models):
                model_data = subset[subset['model'] == model]
                pr = model_data['AUC-PR'].values[0]
                ax2.bar(idx + 0.2, model_data['AUC-PR'], width=0.4, label=None, 
                        color=model_color_map[model], hatch='//', alpha=0.7, edgecolor='black')
                ax2.text(idx + 0.28, pr, f"{pr*100:.3f}%", ha='center', va='bottom', fontsize=8)

            ax.set_title(f"Pattern: {pattern} | Cutoff: {cutoff}")
            ax.set_xticks([])
            ax.set_xticklabels([])
            ax.set_ylabel("AUC-ROC")
            ax2.set_ylabel("AUC-PR")

    # Add legend only once
    handles = [plt.Rectangle((0,0),1,1, color=model_color_map[m]) for m in models]
    fig.legend(handles, models, loc='upper center', ncol=len(models), title='Models - '+dataset, fontsize=12, title_fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig('../results/'+dataset+'_AUC-ROC_AUC-PR.pdf')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

datasets = ['HI-Small', 'LI-Large']
for dataset in datasets:
    results_plot_df = results_df[(results_df['dataset']==dataset)]

    # Ensure consistent aesthetics
    sns.set(style="whitegrid")

    # Assume results_plot_df is already defined
    # Extract unique patterns and cutoffs
    patterns = results_plot_df['pattern'].unique()
    cutoffs = results_plot_df['cutoff'].unique()
    models = [pretty(k) for k in [
        'flowscope',
        'gargaml_u', 'gargaml_d',
        'gargaml_tree_u', 'gargaml_boost_u',
        'gargaml_tree_d', 'gargaml_boost_d',
    ]]
    palette = sns.color_palette("tab10", len(models)+1)
    del (palette[1])

    # Create a mapping from model names to colors
    model_color_map = dict(zip(models, palette))

    # Create the subplot grid
    fig, axes = plt.subplots(len(patterns), len(cutoffs), figsize=(7 * len(cutoffs), 3 * len(patterns)), squeeze=False)

    for i, pattern in enumerate(patterns):
        for j, cutoff in enumerate(cutoffs):
            ax = axes[i, j]
            subset = results_plot_df[(results_plot_df['pattern'] == pattern) & (results_plot_df['cutoff'] == cutoff)]
            ylim_max = subset['AUC-ROC'].max()*1.1
            ax.set_ylim(0, ylim_max)
            # Primary bar plot for AUC-ROC
            for idx, model in enumerate(models):
                model_data = subset[subset['model'] == model]
                roc = model_data['AUC-ROC'].values[0]
                ax.bar(idx - 0.2, model_data['AUC-ROC'], width=0.4, label=model if i == 0 and j == 0 else "", 
                    color=model_color_map[model], edgecolor='black')
                ax.text(idx - 0.23, roc-0.001, f"{roc*100:.1f}%", ha='center', va='bottom', fontsize=8)
            
            # Twin axis for AUC-PR
            ylim_max_2 = subset['AUC-PR'].max()*1.1
            ax2 = ax.twinx()
            ax2.set_ylim(0, ylim_max_2)
            for idx, model in enumerate(models):
                model_data = subset[subset['model'] == model]
                pr = model_data['AUC-PR'].values[0]
                ax2.bar(idx + 0.2, model_data['AUC-PR'], width=0.4, label=None, 
                        color=model_color_map[model], hatch='//', alpha=0.7, edgecolor='black')
                ax2.text(idx + 0.28, pr, f"{pr*100:.3f}%", ha='center', va='bottom', fontsize=8)

            ax.set_title(f"Pattern: {pattern} | Cutoff: {cutoff}", fontsize=20)
            ax.set_xticks([])
            ax.set_xticklabels([])
            #ax.set_xticks(range(len(models)))
            #ax.set_xticklabels(models, rotation=45, ha='right')
            ax.set_ylabel("AUC-ROC")
            ax2.set_ylabel("AUC-PR")

    # Add legend only once
    handles = [plt.Rectangle((0,0),1,1, color=model_color_map[m]) for m in models]
    fig.legend(handles, models, loc='upper center', ncol=len(models), title='Models - '+dataset, fontsize=21, title_fontsize=28)
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig('../results/'+dataset+'_AUC-ROC_AUC-PR_full.pdf')

In [ ]:
import pandas as pd
import numpy as np

def generate_latex_table(results_df, dataset='HI-Small'):
    # Define the ordering and exact naming conventions
    patterns_raw = ['Is Laundering', 'GATHER-SCATTER', 'SCATTER-GATHER']
    patterns_clean = ['Is Laundering', 'Gather-Scatter', 'Scatter-Gather']
    cutoffs = ['0.1', '0.5', '0.9']
    # Precision@K comes from the tidy metrics file merged in above.
    metrics = ['Precision', 'F1-score', 'AUC-ROC', 'AUC-PR', 'P@%d' % K_IBM]
    headers = ['Precision', 'F1', 'AUC-ROC', 'AUC-PR', 'P@%d' % K_IBM]
    
    models = [pretty(k) for k in [
        'gargaml_u', 'gargaml_d',
        'gargaml_tree_u', 'gargaml_boost_u',
        'gargaml_tree_d', 'gargaml_boost_d',
    ]]
    

    # Filter the dataframe for the current table
    df = results_df[
        (results_df['dataset'] == dataset) & 
        (results_df['pattern'].isin(patterns_raw)) & 
        (results_df['cutoff'].isin(cutoffs))
    ].copy()

    # Scale metrics by 100 as done in the plot
    df['AUC-ROC'] = df['AUC-ROC'] * 100
    df['AUC-PR'] = df['AUC-PR'] * 100

    # Pre-calculate maximums for bolding
    max_vals = {}
    for p in patterns_raw:
        max_vals[p] = {}
        for c in cutoffs:
            subset = df[(df['pattern'] == p) & (df['cutoff'] == c)]
            max_vals[p][c] = {
                m: (subset[m].max() if not subset.empty else -1) for m in metrics
            }

    # One decimal for the two percentage-scaled columns, three for the rest.
    decimals = {'AUC-ROC': 1, 'AUC-PR': 3}

    # Start building the LaTeX string
    n_cols = 1 + len(cutoffs) * len(metrics)   # model column + one per metric per cutoff
    latex = []
    latex.append(r"\begin{table*}[t]")
    latex.append(r"\centering")
    latex.append(r"\color{red}")
    latex.append(rf"\caption{{Performance comparison (AUC-ROC and AUC-PR) across graph patterns and cutoffs for the {dataset} dataset.}}")
    latex.append(rf"\label{{tab:results {dataset.lower().replace('-', ' ')}}}")
    latex.append(r"\tiny % Slightly smaller font to ensure fit")
    latex.append(r"\begin{tabular}{l" + (" " + "c" * len(metrics)) * len(cutoffs) + "}")
    latex.append(r"\toprule")
    latex.append(
        r"\multirow{2}{*}{\textbf{Model}} & "
        + " & ".join(rf"\multicolumn{{{len(metrics)}}}{{c}}{{\textbf{{Cutoff {c}}}}}" for c in cutoffs)
        + r" \\"
    )
    latex.append(" ".join(
        rf"\cmidrule(lr){{{2 + j * len(metrics)}-{1 + (j + 1) * len(metrics)}}}"
        for j in range(len(cutoffs))
    ))
    latex.append("& " + " & ".join(headers * len(cutoffs)) + r" \\")

    # Iterate through patterns to build rows. The pattern is a full-width
    # \multicolumn sub-title instead of its own rotated \multirow column.
    for i, pattern_raw in enumerate(patterns_raw):
        latex.append(r"\midrule")
        latex.append(rf"\multicolumn{{{n_cols}}}{{l}}{{\textit{{{patterns_clean[i]}}}}} \\")
        latex.append(r"\midrule")
        
        for model in models:
            row_str = f"{model:<21}"
            
            for cutoff in cutoffs:
                row_data = df[(df['pattern'] == pattern_raw) & (df['cutoff'] == cutoff) & (df['model'] == model)]
                
                if row_data.empty or pd.isna(row_data['AUC-ROC'].values[0]):
                    row_str += " & " + " & ".join(["-"] * len(metrics))
                else:
                    for m in metrics:
                        value = row_data[m].values[0]
                        if pd.isna(value):
                            row_str += " & -"
                            continue
                        # Bold logic: compare to max_val with a small tolerance for floating point issues
                        cell = f"{value:.{decimals.get(m, 3)}f}"
                        if abs(value - max_vals[pattern_raw][cutoff][m]) < 1e-5:
                            cell = rf"\textbf{{{cell}}}"
                        row_str += f" & {cell}"
            
            row_str += r" \\"
            latex.append(row_str)

    latex.append(r"\bottomrule")
    latex.append(r"\end{tabular}")
    latex.append(r"\end{table*}")

    return "\n".join(latex)

# --- Example Usage ---
# latex_output = generate_latex_table(results_df, dataset='HI-Small')
# print(latex_output)
# 
# # You can easily run it for the other dataset too:
# # with open('../results/LI-Large_table.tex', 'w') as f:
# #     f.write(generate_latex_table(results_df, dataset='LI-Large'))

In [ ]:
print(generate_latex_table(results_df, dataset='LI-Large'))

In [ ]:
results_df